# Week 1 — LLM inference flow (hands-on)

Goal: connect **each stage** of inference to **real tensors** you can inspect: tokenization, embedding lookup, prefill (full prompt through the stack), then autoregressive decode (one new token per step).

**Model:** [`WithinUsAI/Gemma3-Prompt.Coder.Uncensored.270m-GGUF`](https://huggingface.co/WithinUsAI/Gemma3-Prompt.Coder.Uncensored.270m-GGUF) — Gemma-3-scale **GGUF** weights loaded through **Transformers** (`gguf_file=...`). First load **de-quantizes** tensors into a normal PyTorch causal LM (one-time cost; not llama.cpp-native GGUF speed). Requires the **`gguf`** package (included in this env).

**Note:** This community GGUF may warn that tokenizer metadata does not match Gemma’s usual class; if outputs look wrong, inspect tokens with `convert_ids_to_tokens`.

**Hardware:** RTX 3050 Ti 4GB — use `bfloat16` on GPU and a modest `max_new_tokens` if you hit OOM.

## 0. Environment check

In [1]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM total (MiB): {props.total_memory // (1024 ** 2)}")

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
print("preferred activations dtype:", dtype)

torch: 2.6.0+cu124
cuda available: True
device: NVIDIA GeForce RTX 3050 Ti Laptop GPU
VRAM total (MiB): 4095
preferred activations dtype: torch.bfloat16


## 1. Load tokenizer + causal LM (GGUF via Transformers)

Inference pipeline (high level):

1. **Tokenize** — text → integer token IDs (`input_ids`).
2. **Embed** — IDs → continuous vectors (lookup + optional scaling).
3. **Transformer blocks** — causal self-attention + MLP, producing hidden states and (for generation) **logits** over the vocabulary at each position.
4. **Decode loop** — sample or greedy-pick the next token, append to context, repeat until stop criteria.

Pick **one** `GGUF_FILE` from the repo (e.g. `Q4_K_M` for size/quality balance). Hub id + filename must match what you [downloaded](https://huggingface.co/WithinUsAI/Gemma3-Prompt.Coder.Uncensored.270m-GGUF/tree/main).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "WithinUsAI/Gemma3-Prompt.Coder.Uncensored.270m-GGUF"
GGUF_FILE = "Gemma-3-Prompt-Coder-270m-it-Uncensored.Q4_K_M.gguf"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, gguf_file=GGUF_FILE)
if tokenizer.pad_token is None and tokenizer.eos_token is not None:
    tokenizer.pad_token = tokenizer.eos_token

device_map = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    gguf_file=GGUF_FILE,
    dtype=dtype,
    device_map=device_map,
)
model.eval()

print("tokenizer vocab size:", tokenizer.vocab_size)
print("model dtype:", next(model.parameters()).dtype)

## 2. Prefill: shape tour

Run the **prompt** through the model **once** (`use_cache=True` stores KV for faster decoding). Inspect `logits` — shape is roughly `(batch, seq_len, vocab_size)`.

In [ ]:
prompt = "Explain one step of autoregressive language model inference in one sentence."
inputs = tokenizer(prompt, return_tensors="pt")
_dev = next(model.parameters()).device
inputs = {k: v.to(_dev) for k, v in inputs.items()}

with torch.no_grad():
    out = model(**inputs, use_cache=True)

logits = out.logits
print("input_ids shape:", inputs["input_ids"].shape)
print("logits shape:", tuple(logits.shape))  # (B, T, V)

last_logits = logits[:, -1, :]
next_id = int(last_logits.argmax(dim=-1).item())
print("greedy next token id:", next_id)
print("greedy next token:", tokenizer.decode([next_id]))

## 3. Full `generate` (prefill + decode)

Use this to compare against your manual shape checks. Lower `max_new_tokens` if you hit OOM.

In [ ]:
gen = model.generate(
    **inputs,
    max_new_tokens=64,
    do_sample=False,
)
print(tokenizer.decode(gen[0], skip_special_tokens=True))

## 4. Week-1 note prompts (fill as you learn)

- **Tokenization:** How does this tokenizer split your prompt? (`tokenizer.convert_ids_to_tokens`)
- **Causal mask:** Why is `logits[t]` only a function of tokens `<= t`?
- **KV cache:** What did `past_key_values` save, and why does it speed up generation?
- **Memory:** For your batch=1 run, what dominates VRAM — weights, activations, or KV cache — if you increase `max_new_tokens`?

Optional: add a cell that prints `out.past_key_values` structure (layer count, tensor shapes) after prefill.